# vLLM benchmark: Qwen2.5-1.5B-Instruct on a cloud GPU

Part of [serve_llm](https://github.com/<your-github-username>/serve_llm): the same model served with
llama.cpp on a laptop CPU and with vLLM here, measured with the same harness (`bench/run_bench.py`).

**Before running**
- **Kaggle:** Settings → Accelerator → **GPU T4 x2** (one GPU is used), and Settings → **Internet on**.
  vLLM does not support the P100 option.
- **Colab:** Runtime → Change runtime type → **T4 GPU**.
- Run the cells in order. Installing vLLM takes a few minutes; if the notebook asks to restart
  after the install, restart and continue from step 3.

**What it measures**
1. Concurrency sweep (1 → 32) at FP16: continuous batching under load.
2. Prefix caching on vs. off, with prompts that share a ~1.8k-token system prompt.
3. *(Optional)* the official AWQ 4-bit model, same sweep.

Results are written as JSON to `results/`; download them and commit them to the repo's `results/`.

## 1. Check the GPU

In [2]:
!nvidia-smi

Mon Oct  5 09:02:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install vLLM and httpx

In [5]:
%pip install -q vllm httpx

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.0 which is incompatible.
pylibcudf-cu12 26.2.1 requires cuda-python<13.0,>=12.9.2, but you have cuda-python 13.4.1 which is incompatible.
numba-cuda 0.22.2 requires cuda-core<1.0.0,>=0.3.2, but you have cuda-core 1.2.1 which is incompatible.
rmm-cu12 26.2.0 requires cuda-python<13.0,>=12.9.2, but you have cuda-python 13.4.1 which is incompatible.
cuml-cu12 26.2.0 requires cuda-python<13.0,>=12.9.2, but you have cuda-python 13.4.1 which is incompatible.
cuml-cu12 26.2.0 requires cuda-toolkit[cublas,cufft,curand,cusolver,cusparse]==12.*, but you have cuda-toolkit 13.0.3.0 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
pylibraft-cu12 26.2.0 requires cuda-python<13.0,>=12

In [10]:
%pip uninstall -y torchaudio

Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
Note: you may need to restart the kernel to use updated packages.


## 3. Configuration

In [6]:
import importlib.metadata
import json
import os
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
AWQ_MODEL = "Qwen/Qwen2.5-1.5B-Instruct-AWQ"
SERVED_NAME = "qwen2.5-1.5b-instruct"  # same model name the laptop runs use
PORT = 8000
BASE_URL = f"http://127.0.0.1:{PORT}/v1"

GITHUB_USER = "<your-github-username>"  # used to download the harness; or upload the files
REPO_RAW = f"https://raw.githubusercontent.com/{GITHUB_USER}/serve_llm/main"

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
RESULTS = WORK / "results"
RESULTS.mkdir(exist_ok=True)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # one GPU, even on a "T4 x2" machine

GPU_NAME = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                          capture_output=True, text=True).stdout.splitlines()[0].strip()
GPU_TAG = GPU_NAME.lower().replace("nvidia", "").replace("tesla", "").strip().replace(" ", "-")
VLLM_VERSION = importlib.metadata.version("vllm")
print(f"GPU: {GPU_NAME} (tag '{GPU_TAG}') · vLLM {VLLM_VERSION} · results -> {RESULTS}")

GPU: Tesla T4 (tag 't4') · vLLM 0.31.0 · results -> /kaggle/working/results


## 4. Get the benchmark harness

Uses `run_bench.py` and `prompts.jsonl` if you uploaded them (working directory or an attached
Kaggle dataset); otherwise downloads them from GitHub using `GITHUB_USER`.

In [7]:
def find_or_download(name: str) -> Path:
    candidates = [WORK / name]
    if Path("/kaggle/input").exists():
        candidates += list(Path("/kaggle/input").rglob(name))
    for path in candidates:
        if path.exists():
            print(f"using {path}")
            return path
    if "<" in GITHUB_USER:
        raise SystemExit("Set GITHUB_USER in step 3, or upload run_bench.py and prompts.jsonl")
    dest = WORK / name
    urllib.request.urlretrieve(f"{REPO_RAW}/bench/{name}", dest)
    print(f"downloaded {dest}")
    return dest


RUN_BENCH = find_or_download("run_bench.py")
PROMPTS = find_or_download("prompts.jsonl")

using /kaggle/input/datasets/hamzahtariq/serve-llm/run_bench.py
using /kaggle/input/datasets/hamzahtariq/serve-llm/prompts.jsonl


## 5. Helpers: start / stop the vLLM server, run the benchmark

In [8]:
def start_server(model: str, *extra: str, log_name: str) -> subprocess.Popen:
    """Start `vllm serve` in the background and wait until /v1/models answers."""
    log_path = WORK / f"{log_name}.log"
    cmd = ["vllm", "serve", model, "--port", str(PORT), "--served-model-name", SERVED_NAME,
           "--dtype", "half",                  # the T4 has no bf16 support
           "--max-model-len", "4096",          # same context size as the laptop runs
           "--gpu-memory-utilization", "0.90",
           "--enable-prompt-tokens-details",   # report cached_tokens per request
           *extra]
    print(" ".join(cmd))
    with open(log_path, "w") as log:  # the child keeps its own handle to the file
        proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
    deadline = time.time() + 900
    while time.time() < deadline:
        if proc.poll() is not None:
            raise RuntimeError(f"vllm exited early; see {log_path}")
        try:
            urllib.request.urlopen(f"{BASE_URL}/models", timeout=2)
            print("server ready")
            return proc
        except OSError:
            time.sleep(5)
    proc.kill()
    raise RuntimeError(f"vllm not ready after 15 min; see {log_path}")


def stop_server(proc: subprocess.Popen) -> None:
    proc.terminate()
    try:
        proc.wait(timeout=60)
    except subprocess.TimeoutExpired:
        proc.kill()
        proc.wait()
    time.sleep(10)  # give the GPU memory time to be released


def bench(label: str, experiment: str, *args: str, server_config: dict) -> None:
    cmd = [sys.executable, str(RUN_BENCH), "--base-url", BASE_URL, "--model", SERVED_NAME,
           "--label", label, "--experiment", experiment, "--prompts", str(PROMPTS),
           "--server-config", json.dumps(server_config),
           "--out", str(RESULTS / f"{label}.json"), *args]
    subprocess.run(cmd, check=True)


def server_config(model: str, **kw) -> dict:
    return {"engine": "vllm", "version": VLLM_VERSION, "gpu": GPU_NAME, "model": model,
            "dtype": "float16", "max_model_len": 4096, **kw}

## 6. Experiment 1: concurrency sweep (FP16)

The same 40 prompts and 256 output tokens per request as the laptop baseline, at 1 → 32
concurrent requests. Prefix caching is left on (vLLM's default); the harness tags every request
so only the chat-template prefix can be reused.

In [11]:
proc = start_server(MODEL, log_name="vllm-fp16")
try:
    bench(f"vllm-{GPU_TAG}-fp16", "load_vllm", "--concurrency", "1,2,4,8,16,32",
          server_config=server_config(MODEL, prefix_caching=True))
finally:
    stop_server(proc)

vllm serve Qwen/Qwen2.5-1.5B-Instruct --port 8000 --served-model-name qwen2.5-1.5b-instruct --dtype half --max-model-len 4096 --gpu-memory-utilization 0.90 --enable-prompt-tokens-details
server ready
[vllm-t4-fp16] concurrency 1: 40 requests ...
[vllm-t4-fp16] c=1  throughput 57.6 tok/s  p50 TTFT 0.03s  p95 latency 5.3s  errors 0
[vllm-t4-fp16] concurrency 2: 40 requests ...
[vllm-t4-fp16] c=2  throughput 126.5 tok/s  p50 TTFT 0.05s  p95 latency 4.2s  errors 0
[vllm-t4-fp16] concurrency 4: 40 requests ...
[vllm-t4-fp16] c=4  throughput 242.0 tok/s  p50 TTFT 0.06s  p95 latency 4.6s  errors 0
[vllm-t4-fp16] concurrency 8: 40 requests ...
[vllm-t4-fp16] c=8  throughput 441.7 tok/s  p50 TTFT 0.08s  p95 latency 5.1s  errors 0
[vllm-t4-fp16] concurrency 16: 40 requests ...
[vllm-t4-fp16] c=16  throughput 661.3 tok/s  p50 TTFT 0.15s  p95 latency 5.4s  errors 0
[vllm-t4-fp16] concurrency 32: 40 requests ...
[vllm-t4-fp16] c=32  throughput 832.9 tok/s  p50 TTFT 0.38s  p95 latency 7.2s  errors 0

## 7. Experiment 2: prefix caching on vs. off

Every request shares one ~1.8k-token system prompt and asks a different short question
(12 questions × 4 repeats, 64 output tokens). With prefix caching the shared prefix is computed
once and reused, so time to first token should drop sharply. The server restarts between runs.

In [12]:
for enabled in (True, False):
    flag = "--enable-prefix-caching" if enabled else "--no-enable-prefix-caching"
    state = "on" if enabled else "off"
    proc = start_server(MODEL, flag, log_name=f"vllm-prefix-{state}")
    try:
        bench(f"vllm-{GPU_TAG}-prefix-{state}", "load_prefix_caching",
              "--prompt-set", "shared_prefix", "--repeat", "4", "--max-tokens", "64",
              "--concurrency", "1,4,16",
              server_config=server_config(MODEL, prefix_caching=enabled))
    finally:
        stop_server(proc)

vllm serve Qwen/Qwen2.5-1.5B-Instruct --port 8000 --served-model-name qwen2.5-1.5b-instruct --dtype half --max-model-len 4096 --gpu-memory-utilization 0.90 --enable-prompt-tokens-details --enable-prefix-caching
server ready
[vllm-t4-prefix-on] concurrency 1: 48 requests ...
[vllm-t4-prefix-on] c=1  throughput 56.1 tok/s  p50 TTFT 0.06s  p95 latency 1.2s  errors 0
[vllm-t4-prefix-on] concurrency 4: 48 requests ...
[vllm-t4-prefix-on] c=4  throughput 178.3 tok/s  p50 TTFT 0.14s  p95 latency 1.5s  errors 0
[vllm-t4-prefix-on] concurrency 16: 48 requests ...
[vllm-t4-prefix-on] c=16  throughput 408.3 tok/s  p50 TTFT 0.34s  p95 latency 2.6s  errors 0
Wrote /kaggle/working/results/vllm-t4-prefix-on.json
vllm serve Qwen/Qwen2.5-1.5B-Instruct --port 8000 --served-model-name qwen2.5-1.5b-instruct --dtype half --max-model-len 4096 --gpu-memory-utilization 0.90 --enable-prompt-tokens-details --no-enable-prefix-caching
server ready
[vllm-t4-prefix-off] concurrency 1: 48 requests ...
[vllm-t4-prefi

## 8. (Optional) Experiment 3: AWQ 4-bit model

Qwen's official AWQ checkpoint of the same model, same sweep as Experiment 1: the GPU-side
counterpart to the laptop's GGUF quantization study. Skip this cell if you are short on GPU time.

In [13]:
proc = start_server(AWQ_MODEL, log_name="vllm-awq")
try:
    bench(f"vllm-{GPU_TAG}-awq", "load_vllm", "--concurrency", "1,2,4,8,16,32",
          server_config=server_config(AWQ_MODEL, quantization="awq", prefix_caching=True))
finally:
    stop_server(proc)

vllm serve Qwen/Qwen2.5-1.5B-Instruct-AWQ --port 8000 --served-model-name qwen2.5-1.5b-instruct --dtype half --max-model-len 4096 --gpu-memory-utilization 0.90 --enable-prompt-tokens-details
server ready
[vllm-t4-awq] concurrency 1: 40 requests ...
[vllm-t4-awq] c=1  throughput 124.3 tok/s  p50 TTFT 0.03s  p95 latency 2.2s  errors 0
[vllm-t4-awq] concurrency 2: 40 requests ...
[vllm-t4-awq] c=2  throughput 254.7 tok/s  p50 TTFT 0.04s  p95 latency 2.2s  errors 0
[vllm-t4-awq] concurrency 4: 40 requests ...
[vllm-t4-awq] c=4  throughput 472.2 tok/s  p50 TTFT 0.05s  p95 latency 2.5s  errors 0
[vllm-t4-awq] concurrency 8: 40 requests ...
[vllm-t4-awq] c=8  throughput 814.8 tok/s  p50 TTFT 0.08s  p95 latency 3.0s  errors 0
[vllm-t4-awq] concurrency 16: 40 requests ...
[vllm-t4-awq] c=16  throughput 1107.8 tok/s  p50 TTFT 0.15s  p95 latency 3.3s  errors 0
[vllm-t4-awq] concurrency 32: 40 requests ...
[vllm-t4-awq] c=32  throughput 1321.4 tok/s  p50 TTFT 0.39s  p95 latency 4.8s  errors 0
Wrot

## 9. Summary

In [14]:
for path in sorted(RESULTS.glob("*.json")):
    run = json.loads(path.read_text())
    n = run["settings"]["num_prompts"]
    print(f"\n{run['label']}  ({run['experiment']}, {n} requests/level)")
    print(f"{'conc':>5} {'tok/s':>8} {'TTFT p50':>9} {'TTFT p95':>9} {'lat p95':>8} "
          f"{'cached':>8} {'errors':>6}")
    for lv in run["levels"]:
        print(f"{lv['concurrency']:>5} {lv['throughput_tps'] or 0:>8.1f} "
              f"{lv['ttft_p50_s'] or 0:>8.2f}s {lv['ttft_p95_s'] or 0:>8.2f}s "
              f"{lv['latency_p95_s'] or 0:>7.1f}s {lv['cached_tokens_total']:>8} {lv['errors']:>6}")


vllm-t4-awq  (load_vllm, 40 requests/level)
 conc    tok/s  TTFT p50  TTFT p95  lat p95   cached errors
    1    124.3     0.03s     0.09s     2.2s      640      0
    2    254.7     0.04s     0.10s     2.2s      640      0
    4    472.2     0.05s     0.27s     2.5s      640      0
    8    814.8     0.08s     0.39s     3.0s      640      0
   16   1107.8     0.15s     0.38s     3.3s      640      0
   32   1321.4     0.39s     0.40s     4.8s      640      0

vllm-t4-fp16  (load_vllm, 40 requests/level)
 conc    tok/s  TTFT p50  TTFT p95  lat p95   cached errors
    1     57.6     0.03s     0.09s     5.3s      640      0
    2    126.5     0.05s     0.10s     4.2s      640      0
    4    242.0     0.06s     0.26s     4.6s      640      0
    8    441.7     0.08s     0.39s     5.1s      640      0
   16    661.3     0.15s     0.40s     5.4s      640      0
   32    832.9     0.38s     0.39s     7.2s      640      0

vllm-t4-prefix-off  (load_prefix_caching, 48 requests/level)
 conc  

## 10. Download the results

- **Kaggle:** the files are in the notebook's **Output** panel (`results/` and `vllm_results.zip`).
- **Colab:** the cell below also triggers a browser download.

Then copy the JSON files into the repo's `results/` folder and run `uv run python -m serve_llm.plot`.

In [15]:
import shutil

archive = shutil.make_archive(str(WORK / "vllm_results"), "zip", RESULTS)
print(f"wrote {archive}")
try:
    from google.colab import files  # only exists on Colab

    files.download(archive)
except ImportError:
    pass

wrote /kaggle/working/vllm_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>